# 03 — Treaty Anchor Analysis

Examine treaty anchor similarity scores, country-treaty heatmaps, and semantic drift.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import load_config
from src.data.load_ungdc import load_ungdc
from src.data.segment import segment_by_keywords
from src.data.preprocess import preprocess_corpus
from src.data.load_treaties import load_treaty_anchors, get_all_anchors_flat

cfg = load_config('../config.yaml')

# Show treaty anchors
treaty_anchors = load_treaty_anchors()
for treaty, passages in treaty_anchors.items():
    print(f"\n=== {treaty.upper()} ===")
    print(passages[0][:200] + '...')

In [ ]:
# Load or compute anchor scores
import os
from pathlib import Path

anchor_scores_path = cfg.output_dir / 'embeddings' / 'anchor_scores.csv'

if anchor_scores_path.exists():
    anchor_scores = pd.read_csv(anchor_scores_path)
    print(f"Loaded pre-computed anchor scores: {len(anchor_scores)} rows")
else:
    print("Computing anchor scores (may take a moment)...")
    corpus_df = load_ungdc(data_dir=str(cfg.data_dir), synthetic_mode=cfg.synthetic_mode,
                           year_start=cfg.focus_start, year_end=cfg.focus_end)
    corpus_df = preprocess_corpus(corpus_df)
    segments_df = segment_by_keywords(corpus_df, window_size=cfg.keyword_window_size)
    if segments_df.empty:
        segments_df = corpus_df
    
    from src.analysis.embeddings import embed_treaty_anchors, compute_country_year_anchor_scores
    anchor_embeddings = embed_treaty_anchors(treaty_anchors, model_name=cfg.embedding_model)
    anchor_scores = compute_country_year_anchor_scores(segments_df, anchor_embeddings)
    anchor_scores_path.parent.mkdir(parents=True, exist_ok=True)
    anchor_scores.to_csv(anchor_scores_path, index=False)

anchor_scores.head()

In [ ]:
# Country-treaty heatmap
from src.viz.comparisons import plot_anchor_similarity_heatmap
plot_anchor_similarity_heatmap(anchor_scores, top_n=25)
plt.show()

In [ ]:
# Group-level anchor similarity over time
from src.groups import get_group_members
from src.viz.temporal import plot_treaty_anchor_similarity_over_time

groups = {
    'P5': get_group_members('p5'),
    'NAC': get_group_members('nac'),
    'NAM': get_group_members('nam'),
    'Gulf': get_group_members('gulf_states'),
}

# Add mean anchor sim column
score_cols = [c for c in anchor_scores.columns if c.endswith('_score')]
anchor_scores['mean_anchor_sim'] = anchor_scores[score_cols].mean(axis=1)

plot_treaty_anchor_similarity_over_time(anchor_scores, groups)
plt.show()

In [ ]:
# Top countries by NPT anchor similarity
if 'npt_score' in anchor_scores.columns:
    top_npt = (
        anchor_scores.groupby('country_code')['npt_score']
        .mean()
        .sort_values(ascending=False)
        .head(15)
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    top_npt.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('Mean NPT Anchor Similarity')
    ax.set_title('Top 15 Countries by NPT Rhetorical Similarity')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    print(top_npt)

In [ ]:
# TPNW vs NPT similarity gap (pro-ban vs NPT-only rhetoric)
if 'tpnw_score' in anchor_scores.columns and 'npt_score' in anchor_scores.columns:
    agg = anchor_scores.groupby('country_code')[['tpnw_score', 'npt_score']].mean()
    agg['tpnw_npt_gap'] = agg['tpnw_score'] - agg['npt_score']
    top_gap = agg.nlargest(15, 'tpnw_npt_gap')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    top_gap['tpnw_npt_gap'].plot(kind='barh', ax=ax, color='#2ca02c')
    ax.set_xlabel('TPNW − NPT Similarity Gap')
    ax.set_title('Countries with Highest TPNW vs NPT Rhetorical Gap')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()